# OPT-6.7B W/A BFP Baseline

Quantize every `nn.Linear` weight and activation to configurable Block Floating Point (BFP), run the linear operation, and return FP16 output for the remaining model operations. One run automatically sweeps **BFP8 through BFP4** with block size 32 and shared signed E5. Each format reloads the original FP16 checkpoint before quantization.

This is fake quantization: values are quantized and dequantized onto FP16 tensors before `F.linear`. OPT may tie `lm_head.weight` to the input embedding. When tied storage is detected, this notebook clones `lm_head.weight` before Linear quantization so the embedding lookup remains original FP16.

In [ ]:
%pip install -q "transformers==5.13.1" "datasets==4.0.0" accelerate sentencepiece tqdm

In [ ]:
import gc
import json
import platform
import time
import zipfile
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "facebook/opt-6.7b"
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
SPLIT = "test"
CONTEXT_LENGTH = 2048
STRIDE = 2048
DROP_REMAINDER = True
EVALUATION_PROTOCOL = "non_overlapping_2048_drop_remainder"
BASELINE_PPL = 10.860297203063965
BASELINE_TOKEN_COUNTS = {
    "source_input_tokens": 287645,
    "used_input_tokens": 286720,
    "dropped_input_tokens": 925,
    "evaluated_blocks": 140,
    "evaluated_tokens": 286580,
}
MANTISSA_BITS_SWEEP = (7, 6, 5, 4, 3)


@dataclass(frozen=True)
class BFPConfig:
    block_size: int = 32
    shared_exponent_bits: int = 5
    mantissa_bits: int = 7
    rounding: str = "nearest"
    weight_chunk_rows: int = 128
    activation_chunk_rows: int = 2048
    quantize_lm_head: bool = True

    def validate(self):
        if self.block_size <= 0:
            raise ValueError("block_size must be positive.")
        if self.shared_exponent_bits < 2:
            raise ValueError("shared_exponent_bits must be at least 2.")
        if self.mantissa_bits <= 0:
            raise ValueError("mantissa_bits must be positive.")
        if self.rounding not in {"nearest", "trunc"}:
            raise ValueError("rounding must be 'nearest' or 'trunc'.")


BFP = BFPConfig()
BFP.validate()
OUTPUT_DIR = Path("bfp-s2048-results")
ARCHIVE_PATH = Path("opt-6.7b-bfp8-bfp4-s2048.zip")

if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires an NVIDIA CUDA GPU.")

torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32 = False
print(f"Base config: {BFP}")
print(f"Sweep: {[f'BFP{1 + bits}' for bits in MANTISSA_BITS_SWEEP]}")

## BFP convention and Linear replacement

Blocks are contiguous along the last dimension. Weight rows and token activation vectors are partitioned independently. Shared exponents are signed E5 integers; mantissas use round-to-nearest-even by default.

In [ ]:
def _quantize_bfp_rows(rows, config):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size

    if padding:
        flat = F.pad(flat, (0, padding))

    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    max_abs = blocks.abs().amax(dim=-1, keepdim=True)
    safe_max = max_abs.clamp_min(torch.finfo(torch.float32).tiny)
    shared_exp = torch.floor(torch.log2(safe_max))

    exp_min = -(1 << (config.shared_exponent_bits - 1))
    exp_max = (1 << (config.shared_exponent_bits - 1)) - 1
    shared_exp = shared_exp.clamp(exp_min, exp_max)
    shared_exp = torch.where(max_abs == 0, torch.zeros_like(shared_exp), shared_exp)

    step = torch.pow(2.0, shared_exp - (config.mantissa_bits - 1))
    mantissa = blocks / step
    mantissa = torch.round(mantissa) if config.rounding == "nearest" else torch.trunc(mantissa)
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = mantissa.clamp(-mantissa_max, mantissa_max)

    dequantized = (mantissa * step).reshape(flat.size(0), padded_width)
    dequantized = dequantized[:, :width].reshape(original_shape)
    return dequantized.to(rows.dtype)


def quantize_bfp(tensor, config, chunk_rows):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    if flat.size(0) <= chunk_rows:
        return _quantize_bfp_rows(tensor, config)

    output = torch.empty_like(flat)
    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        output[start:end] = _quantize_bfp_rows(flat[start:end], config)
    return output.reshape_as(tensor)


@torch.no_grad()
def quantize_weight_in_place(weight, config):
    for start in range(0, weight.size(0), config.weight_chunk_rows):
        end = min(start + config.weight_chunk_rows, weight.size(0))
        weight[start:end].copy_(_quantize_bfp_rows(weight[start:end], config))


class BFPLinear(nn.Module):
    def __init__(self, linear, config):
        super().__init__()
        self.linear = linear
        self.config = config

    def forward(self, x):
        x_bfp = quantize_bfp(x, self.config, self.config.activation_chunk_rows)
        return F.linear(x_bfp, self.linear.weight, self.linear.bias).to(torch.float16)


def replace_linear_layers(module, config, prefix=""):
    replaced = []
    for name, child in list(module.named_children()):
        full_name = f"{prefix}.{name}" if prefix else name
        if isinstance(child, nn.Linear):
            if full_name == "lm_head" and not config.quantize_lm_head:
                continue
            quantize_weight_in_place(child.weight, config)
            setattr(module, name, BFPLinear(child, config))
            replaced.append(full_name)
        else:
            replaced.extend(replace_linear_layers(child, config, full_name))
    return replaced


def untie_output_head_for_linear_only_quantization(model):
    input_embeddings = model.get_input_embeddings()
    output_embeddings = model.get_output_embeddings()
    if input_embeddings is None or output_embeddings is None:
        raise RuntimeError("OPT input or output embeddings are unavailable.")

    input_weight = input_embeddings.weight
    was_tied = input_weight.data_ptr() == output_embeddings.weight.data_ptr()
    if was_tied:
        output_embeddings.weight = nn.Parameter(
            output_embeddings.weight.detach().clone(),
            requires_grad=output_embeddings.weight.requires_grad,
        )
        model.config.tie_word_embeddings = False

    if input_weight.data_ptr() == output_embeddings.weight.data_ptr():
        raise RuntimeError("lm_head and input embedding storage must be independent.")
    return was_tied, input_weight, output_embeddings.weight


sample = torch.tensor([[0.0, -1.0, 0.5, 1.5]], device="cuda", dtype=torch.float16)
sample_q = _quantize_bfp_rows(sample, BFP)
assert sample_q.shape == sample.shape
assert sample_q.dtype == torch.float16
assert torch.isfinite(sample_q).all()

## Tokenizer, WikiText-2, and evaluator

The public OPT checkpoint requires no Hugging Face token. Transformers v5 may select its unified fast tokenizer backend even when `use_fast=False`; the actual backend is recorded.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False)
print(f"Tokenizer: {tokenizer.__class__.__name__}, is_fast={tokenizer.is_fast}")

dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=SPLIT)
text = "\n\n".join(dataset["text"])
input_ids = tokenizer(text, return_tensors="pt").input_ids
assert input_ids.numel() == BASELINE_TOKEN_COUNTS["source_input_tokens"], input_ids.numel()
print(f"WikiText-2 {SPLIT} tokens: {input_ids.numel():,}")

In [ ]:
@torch.inference_mode()
def evaluate_perplexity(model, input_ids, context_length, stride, drop_remainder):
    if stride != context_length:
        raise ValueError("Non-overlapping evaluation requires stride == context_length.")
    if not drop_remainder:
        raise ValueError("Paper-compatible evaluation requires drop_remainder=True.")
    if context_length > model.config.max_position_embeddings:
        raise ValueError("context_length exceeds the model context window.")

    device = next(model.parameters()).device
    sequence_length = input_ids.size(1)
    usable_length = sequence_length // context_length * context_length
    dropped_tokens = sequence_length - usable_length
    if usable_length == 0:
        raise ValueError("Input does not contain a complete context block.")

    total_nll = 0.0
    total_loss_tokens = 0
    total_blocks = usable_length // context_length

    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize(device)
    start_time = time.perf_counter()

    for begin in tqdm(range(0, usable_length, stride), total=total_blocks, desc="Evaluating"):
        end = begin + context_length
        batch = input_ids[:, begin:end].to(device)
        labels = batch.clone()
        loss = model(batch, labels=labels, use_cache=False).loss
        loss_tokens = labels[:, 1:].numel()
        total_nll += loss.float().item() * loss_tokens
        total_loss_tokens += loss_tokens

    torch.cuda.synchronize(device)
    elapsed_seconds = time.perf_counter() - start_time
    mean_nll = total_nll / total_loss_tokens
    return {
        "mean_nll": mean_nll,
        "perplexity": float(torch.exp(torch.tensor(mean_nll))),
        "source_input_tokens": sequence_length,
        "used_input_tokens": usable_length,
        "dropped_input_tokens": dropped_tokens,
        "evaluated_blocks": total_blocks,
        "evaluated_tokens": total_loss_tokens,
        "elapsed_seconds": elapsed_seconds,
        "tokens_per_second": total_loss_tokens / elapsed_seconds,
        "peak_gpu_memory_gib": torch.cuda.max_memory_allocated(device) / 2**30,
    }

## Run BFP8 through BFP4

Each iteration reloads the original FP16 checkpoint from the Hugging Face cache. The output head is untied only when shared storage is detected, then all 193 Linear layers are independently fake-quantized.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results = []
output_paths = []

for mantissa_bits in MANTISSA_BITS_SWEEP:
    config = replace(BFP, mantissa_bits=mantissa_bits)
    config.validate()
    bfp_bits = 1 + config.mantissa_bits
    output_path = OUTPUT_DIR / f"bfp{bfp_bits}-g{config.block_size}-s2048.json"

    print(f"\n{'=' * 72}")
    print(f"Running BFP{bfp_bits}: {config}")
    print('=' * 72)

    torch.manual_seed(0)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        device_map=0,
        attn_implementation="eager",
    )
    model.eval()
    model.config.use_cache = False

    parameter_dtypes = {p.dtype for p in model.parameters() if p.is_floating_point()}
    parameter_devices = {(p.device.type, p.device.index) for p in model.parameters()}
    assert parameter_dtypes == {torch.float16}, parameter_dtypes
    assert parameter_devices == {("cuda", 0)}, parameter_devices
    assert not getattr(model, "is_quantized", False)
    assert int(model.config.num_hidden_layers) == 32
    assert int(model.config.hidden_size) == int(model.config.word_embed_proj_dim) == 4096
    assert CONTEXT_LENGTH <= int(model.config.max_position_embeddings)

    linear_names_before = [name for name, module in model.named_modules() if isinstance(module, nn.Linear)]
    expected_linear_count = int(model.config.num_hidden_layers) * 6 + 1
    assert expected_linear_count == 193
    assert len(linear_names_before) == expected_linear_count, linear_names_before

    lm_head_was_tied, input_embedding_weight, output_head_weight = (
        untie_output_head_for_linear_only_quantization(model)
    )
    input_embedding_version = input_embedding_weight._version
    assert input_embedding_weight.dtype == torch.float16
    assert input_embedding_weight.data_ptr() != output_head_weight.data_ptr()

    quantized_layers = replace_linear_layers(model, config)
    quantized_layer_count = len(quantized_layers)
    assert quantized_layer_count == expected_linear_count, quantized_layer_count
    assert input_embedding_weight._version == input_embedding_version
    assert input_embedding_weight.dtype == torch.float16
    assert model.get_input_embeddings().weight.data_ptr() != model.get_output_embeddings().linear.weight.data_ptr()
    torch.cuda.empty_cache()

    print(f"Quantized Linear layers: {quantized_layer_count}")
    print(f"lm_head was tied: {lm_head_was_tied}")
    metrics = evaluate_perplexity(model, input_ids, CONTEXT_LENGTH, STRIDE, DROP_REMAINDER)
    for name, expected in BASELINE_TOKEN_COUNTS.items():
        assert metrics[name] == expected, (name, metrics[name], expected)

    result = {
        "model": MODEL_ID,
        "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
        "split": SPLIT,
        "quantization": "W/A BFP fake quantization",
        "format": f"BFP{bfp_bits} (1S{config.mantissa_bits}M + shared E{config.shared_exponent_bits})",
        "bfp_config": asdict(config),
        "sweep_mantissa_bits": list(MANTISSA_BITS_SWEEP),
        "shared_exponent_encoding": "signed integer",
        "linear_output_dtype": "float16",
        "matmul_backend": "torch.nn.functional.linear with dequantized FP16 operands",
        "quantized_linear_layers": quantized_layer_count,
        "expected_opt_linear_layers": expected_linear_count,
        "input_embedding_quantized": False,
        "lm_head_was_tied_before_quantization": lm_head_was_tied,
        "lm_head_tie_handling": (
            "cloned to independent FP16 Parameter before BFP quantization"
            if lm_head_was_tied
            else "already independent in loaded checkpoint"
        ),
        "tokenizer_class": tokenizer.__class__.__name__,
        "tokenizer_use_fast": tokenizer.is_fast,
        "attention_implementation": "eager",
        "context_length": CONTEXT_LENGTH,
        "stride": STRIDE,
        "evaluation_protocol": EVALUATION_PROTOCOL,
        "drop_remainder": DROP_REMAINDER,
        "model_max_position_embeddings": model.config.max_position_embeddings,
        "baseline_perplexity": BASELINE_PPL,
        "delta_perplexity": metrics["perplexity"] - BASELINE_PPL,
        "gpu": torch.cuda.get_device_name(0),
        "cuda": torch.version.cuda,
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        **metrics,
    }

    output_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    results.append(result)
    output_paths.append(output_path)
    print(json.dumps(result, indent=2, ensure_ascii=False))
    print(f"Saved: {output_path.resolve()}")

    del model, quantized_layers, input_embedding_weight, output_head_weight
    gc.collect()
    torch.cuda.empty_cache()

print("\nSweep complete:")
for result in results:
    print(f"{result['format'].split()[0]}: PPL={result['perplexity']:.6f}")

In [ ]:
with zipfile.ZipFile(ARCHIVE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in output_paths:
        archive.write(path, arcname=path.name)

print(f"Created: {ARCHIVE_PATH.resolve()}")
from google.colab import files
files.download(str(ARCHIVE_PATH))